In [0]:
# Create DataFrame from Parquet

# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")


In [0]:
path = f'/Volumes/mano_adb_outlook/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

# Check if the landing_timestamp folder exists
try:
    dbutils.fs.ls(path)
except Exception:
    print(f"No new file available at: {path}")
    dbutils.notebook.exit("No new file available")

if source_file_format == 'csv':
    df2 = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(path)
else:
    df2 = spark.read.format('parquet').load(path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp

df2 = df2.withColumn("landing_timestamp", lit(landing_timestamp)) \
         .withColumn("ingestion_timestamp", current_timestamp())

In [0]:
df2.write.mode('append').saveAsTable(f'mano_adb_outlook.bronze.{table_name}')